# Air Quality Index (AQI) Prediction Using Machine Learning

This notebook predicts the daily AQI for 2025 from past pollutant data.
It uses two models, Random Forest and Gradient Boosting.

**How to run it**

1. Click Runtime, then Run all.
2. When asked, upload your Excel file.
3. Wait for the plots and scores to appear.
4. The predictions file will download at the end.

**Your Excel file needs these columns:** Date, Temperature, Humidity, NO2, SO2, PM2.5

## 1. Import libraries

In [ ]:
# First we bring in the tools we need.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# These two models will make the predictions.
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# These metrics tell us how good the predictions are.
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# This lets us upload and download files in Colab.
from google.colab import files

## 2. Load the data

In [ ]:
# A file picker will open. Choose your Excel file.
uploaded = files.upload()

# Read the first uploaded file into a table.
df = pd.read_excel(list(uploaded.keys())[0])

# Take a quick look at the first rows.
# This confirms the file loaded the right way.
df.head()

## 3. Clean the data

In [ ]:
# Turn the Date column into real dates.
df['Date'] = pd.to_datetime(df['Date'])

# Make sure every measurement column holds numbers.
# Anything that is not a number becomes empty for now.
for col in ['Temperature', 'Humidity', 'NO2', 'SO2', 'PM2.5']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Fill empty cells with the last known value.
df = df.ffill()

# If the very first rows are empty, use the next known value.
df = df.bfill()

## 4. Calculate the AQI

In [ ]:
# This function turns one pollutant reading into a sub index.
# It needs the reading C and the list of breakpoints for that pollutant.
def calculate_sub_aqi(C, breakpoints):
    # Go through each AQI band one by one.
    for bp in breakpoints:
        # Check if the reading falls inside this band.
        if bp['C_low'] <= C <= bp['C_high']:
            # Apply the standard AQI formula.
            return ((bp['I_high'] - bp['I_low']) /
                    (bp['C_high'] - bp['C_low'])) * (C - bp['C_low']) + bp['I_low']
    # No band matched, so we return nothing.
    return None

In [ ]:
# Breakpoints for PM2.5.
# C is the concentration and I is the AQI value for that band.
pm25_breakpoints = [
    {'C_low': 0.0,   'C_high': 12.0,  'I_low': 0,   'I_high': 50},
    {'C_low': 12.1,  'C_high': 35.4,  'I_low': 51,  'I_high': 100},
    {'C_low': 35.5,  'C_high': 55.4,  'I_low': 101, 'I_high': 150},
    {'C_low': 55.5,  'C_high': 150.4, 'I_low': 151, 'I_high': 200},
    {'C_low': 150.5, 'C_high': 250.4, 'I_low': 201, 'I_high': 300},
    {'C_low': 250.5, 'C_high': 500.4, 'I_low': 301, 'I_high': 500}
]

# Breakpoints for NO2.
no2_breakpoints = [
    {'C_low': 0,    'C_high': 53,   'I_low': 0,   'I_high': 50},
    {'C_low': 54,   'C_high': 100,  'I_low': 51,  'I_high': 100},
    {'C_low': 101,  'C_high': 360,  'I_low': 101, 'I_high': 150},
    {'C_low': 361,  'C_high': 649,  'I_low': 151, 'I_high': 200},
    {'C_low': 650,  'C_high': 1249, 'I_low': 201, 'I_high': 300},
    {'C_low': 1250, 'C_high': 2049, 'I_low': 301, 'I_high': 500}
]

# Breakpoints for SO2.
so2_breakpoints = [
    {'C_low': 0,   'C_high': 35,   'I_low': 0,   'I_high': 50},
    {'C_low': 36,  'C_high': 75,   'I_low': 51,  'I_high': 100},
    {'C_low': 76,  'C_high': 185,  'I_low': 101, 'I_high': 150},
    {'C_low': 186, 'C_high': 304,  'I_low': 151, 'I_high': 200},
    {'C_low': 305, 'C_high': 604,  'I_low': 201, 'I_high': 300},
    {'C_low': 605, 'C_high': 1004, 'I_low': 301, 'I_high': 500}
]

In [ ]:
# Work out a sub index for each pollutant, row by row.
df['PM2.5_AQI'] = df['PM2.5'].apply(lambda x: calculate_sub_aqi(x, pm25_breakpoints))
df['NO2_AQI'] = df['NO2'].apply(lambda x: calculate_sub_aqi(x, no2_breakpoints))
df['SO2_AQI'] = df['SO2'].apply(lambda x: calculate_sub_aqi(x, so2_breakpoints))

# The final AQI is the worst of the three sub indices.
df['AQI'] = df[['PM2.5_AQI', 'NO2_AQI', 'SO2_AQI']].max(axis=1)

## 5. Create lag features

In [ ]:
# A lag feature is yesterday's value.
# Pollution today depends a lot on pollution yesterday.
df['PM2.5_lag1'] = df['PM2.5'].shift(1)
df['NO2_lag1'] = df['NO2'].shift(1)
df['SO2_lag1'] = df['SO2'].shift(1)

# The first row has no yesterday, so we drop it.
df.dropna(inplace=True)

## 6. Split into training and testing data

In [ ]:
# Train on 2018 to 2024 data.
train_df = df[df['Date'].dt.year <= 2024]

# Test on 2025 data.
# The model never sees this during training, so the test is fair.
test_df = df[df['Date'].dt.year == 2025].copy()

# These are the inputs the model will learn from.
features = ['PM2.5', 'NO2', 'SO2', 'PM2.5_lag1', 'NO2_lag1', 'SO2_lag1']

# This is the value we want to predict.
target = 'AQI'

# Separate inputs and answers for both sets.
X_train = train_df[features]
y_train = train_df[target]
X_test = test_df[features]
y_test = test_df[target]

## 7. Train the models

In [ ]:
# Random Forest builds many decision trees and averages them.
# It handles curved patterns well and does not overfit easily.
rf_model = RandomForestRegressor(n_estimators=50, random_state=42)
rf_model.fit(X_train, y_train)

# Predict the AQI for 2025.
rf_pred = rf_model.predict(X_test)

In [ ]:
# Gradient Boosting builds small trees one after another.
# Each new tree fixes the mistakes of the last one.
gbr = GradientBoostingRegressor(n_estimators=200, learning_rate=0.03, max_depth=2, random_state=42)
gbr.fit(X_train, y_train)

# Predict the AQI for 2025.
gbr_pred = gbr.predict(X_test)

## 8. Check how good the predictions are

In [ ]:
# This function prints three scores for a model.
def evaluate_model(y_true, y_pred, model_name):
    # MAE is the average size of the error.
    mae = mean_absolute_error(y_true, y_pred)
    # RMSE is like MAE but big mistakes count more.
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    # R2 shows how much of the AQI pattern the model explains.
    # A value close to 1 is better.
    r2 = r2_score(y_true, y_pred)

    print(f"=== {model_name} ===")
    print(f"MAE: {mae:.2f}")
    print(f"RMSE: {rmse:.2f}")
    print(f"R² Score: {r2:.4f} ({r2*100:.2f}%)\n")

# Score both models on the 2025 data.
evaluate_model(y_test, rf_pred, "Random Forest")
evaluate_model(y_test, gbr_pred, "Gradient Boosting")

## 9. Plot the results

In [ ]:
# Line plot of the real AQI against both predictions.
plt.figure(figsize=(12, 6))
plt.plot(test_df['Date'], y_test, label='Actual AQI', linewidth=2)
plt.plot(test_df['Date'], rf_pred, label='Random Forest Prediction', linestyle='--')
plt.plot(test_df['Date'], gbr_pred, label='Gradient Boosting Prediction', linestyle=':')
plt.xlabel("Date")
plt.ylabel("AQI")
plt.title("AQI Prediction: Actual vs Predicted (2025)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# One small scatter plot for each pollutant.
# Blue dots are the real AQI and orange dots are the predicted AQI.
plt.figure(figsize=(15, 5))
for i, pollutant in enumerate(['NO2', 'SO2', 'PM2.5']):
    plt.subplot(1, 3, i + 1)
    plt.scatter(test_df[pollutant], y_test, label='Actual AQI', alpha=0.7)
    plt.scatter(test_df[pollutant], gbr_pred, label='Predicted AQI', alpha=0.7)
    plt.xlabel(pollutant)
    plt.ylabel('AQI')
    plt.title(f'{pollutant} vs AQI')
    plt.legend()
    plt.grid(True)

# Stop the plots from overlapping.
plt.tight_layout()
plt.show()

## 10. Save the predictions

In [ ]:
# Add both predictions as new columns.
test_df['AQI_RF_Predicted'] = rf_pred
test_df['AQI_GBR_Predicted'] = gbr_pred

# Save everything to an Excel file.
test_df.to_excel("Predicted_AQI_2025.xlsx", index=False)

# Download the file to your computer.
files.download("Predicted_AQI_2025.xlsx")